# Guitar Tablature Transcription Training — v2 (Fixed Loss)

**What changed from v1:**
- ❌ v1 used `nn.BCELoss()` on extremely sparse targets (~0.03% positive onsets)
- ❌ Model collapsed to predicting all zeros (low loss but zero recall)
- ✅ v2 uses `BCEWithLogitsLoss(pos_weight=50)` + Focal Loss for onsets
- ✅ v2 uses `BCEWithLogitsLoss(pos_weight=10)` for frames
- ✅ v2 tracks F1/Precision/Recall (not just loss) to detect collapse
- ✅ v2 adds data augmentation (gain jitter, noise)
- ✅ Model returns raw logits (no sigmoid during training)

**Architecture:** Same CRNN (CQT → Conv2D → BiLSTM → onset/frame heads)

**Dataset:** GuitarSet (360 recordings with MIDI+Tab annotations)

**Estimated time:** 4-8 hours on T4, 2-4 hours on V100

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q torch torchaudio librosa mir_eval jams scikit-learn tqdm

In [ ]:
# Download GuitarSet dataset
import os
import subprocess

ANNOT_URL = "https://zenodo.org/records/3371780/files/annotation.zip?download=1"
AUDIO_URL = "https://zenodo.org/records/3371780/files/audio_mono-mic.zip?download=1"
DATA_DIR = "/content/guitarset"

if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)

    # Download annotations (39 MB)
    print("Downloading GuitarSet annotations (~39 MB)...")
    !wget -q --show-progress "{ANNOT_URL}" -O /content/annotation.zip
    !unzip -q /content/annotation.zip -d {DATA_DIR}/
    !rm -f /content/annotation.zip

    # Download mono mic audio (657 MB)
    print("\nDownloading GuitarSet audio (~657 MB)...")
    !wget -q --show-progress "{AUDIO_URL}" -O /content/audio_mono-mic.zip
    !unzip -q /content/audio_mono-mic.zip -d {DATA_DIR}/
    !rm -f /content/audio_mono-mic.zip

    print("\nDataset downloaded!")
else:
    print("Dataset already exists")

# === DIAGNOSTIC: Show exactly what we got ===
print("\n=== All directories ===")
for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for f in sorted(files)[:5]:
            print(f'{subindent}{f}')
        if len(files) > 5:
            print(f'{subindent}... and {len(files)-5} more files')

import glob
wav_files = glob.glob(f'{DATA_DIR}/**/*.wav', recursive=True)
jams_files = glob.glob(f'{DATA_DIR}/**/*.jams', recursive=True)
print(f"\nTotal WAV files: {len(wav_files)}")
print(f"Total JAMS files: {len(jams_files)}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import numpy as np
import librosa
import jams
import random
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Configuration — MUST match inference in guitar_tab_transcriber.py
CONFIG = {
    'sample_rate': 22050,
    'hop_length': 256,
    'n_bins': 84,           # CQT bins (7 octaves x 12)
    'bins_per_octave': 12,

    # Guitar specifics
    'num_strings': 6,
    'num_frets': 20,  # 0-19
    'tuning': [40, 45, 50, 55, 59, 64],  # Standard tuning MIDI notes

    # Training
    'batch_size': 16,
    'learning_rate': 1e-4,
    'num_epochs': 100,
    'chunk_length_sec': 3.0,

    # v2: Loss weights
    'onset_pos_weight': 50.0,   # Positive class weight for onsets
    'frame_pos_weight': 10.0,   # Positive class weight for frames
    'focal_gamma': 2.0,         # Focal loss gamma
}

# MIDI to (string, fret) mapping for standard tuning
def midi_to_tab(midi_note, tuning=CONFIG['tuning']):
    """Convert MIDI note to all possible (string, fret) combinations."""
    positions = []
    for string_idx, open_note in enumerate(tuning):
        fret = midi_note - open_note
        if 0 <= fret < CONFIG['num_frets']:
            positions.append((string_idx, fret))
    return positions

print(f"Output neurons: {CONFIG['num_strings']} x {CONFIG['num_frets']} = {CONFIG['num_strings'] * CONFIG['num_frets']}")
print(f"Onset pos_weight: {CONFIG['onset_pos_weight']}")
print(f"Frame pos_weight: {CONFIG['frame_pos_weight']}")
print(f"Focal gamma: {CONFIG['focal_gamma']}")

## v2 Loss Functions & Metrics

The key fix: **Focal Loss with pos_weight** for onsets, **weighted BCE** for frames.

Why v1 failed: With ~0.03% positive onsets, plain BCELoss converges to predicting all zeros.
That gives val_loss ~0.05 (looks "good") but onset recall = 0.0 (model is dead).

In [ ]:
class FocalBCEWithLogitsLoss(nn.Module):
    """
    Focal Loss + pos_weight for extremely imbalanced binary classification.

    Combines two fixes:
    1. pos_weight: Upweights positive samples (onsets are ~0.03% of targets)
    2. Focal term: Down-weights easy negatives, focuses on hard examples

    This prevents the model from collapsing to predict-all-zeros.
    """

    def __init__(self, pos_weight=50.0, gamma=2.0):
        super().__init__()
        self.pos_weight = pos_weight
        self.gamma = gamma

    def forward(self, logits, targets):
        # BCEWithLogitsLoss with pos_weight (numerically stable)
        pw = torch.tensor([self.pos_weight], device=logits.device, dtype=logits.dtype)
        bce = F.binary_cross_entropy_with_logits(
            logits, targets,
            pos_weight=pw,
            reduction='none'
        )

        # Focal modulation: (1 - p_t)^gamma
        probs = torch.sigmoid(logits)
        pt = torch.where(targets > 0.5, probs, 1 - probs)
        focal_weight = (1 - pt) ** self.gamma

        return (focal_weight * bce).mean()


def compute_onset_f1(logits, targets, threshold=0.5):
    """
    Compute onset F1/precision/recall from logits.

    This is THE critical metric. If recall stays at 0.0 for 5+ epochs,
    the model has collapsed and training should be stopped.
    """
    with torch.no_grad():
        preds = (torch.sigmoid(logits) > threshold).float()
        tp = (preds * targets).sum()
        fp = (preds * (1 - targets)).sum()
        fn = ((1 - preds) * targets).sum()

        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)

        return f1.item(), precision.item(), recall.item()


print('✅ FocalBCEWithLogitsLoss and F1 metrics defined')

# Quick sanity check
loss_fn = FocalBCEWithLogitsLoss(pos_weight=50.0, gamma=2.0)
dummy_logits = torch.randn(2, 6, 20, 100)  # (batch, strings, frets, time)
dummy_targets = torch.zeros(2, 6, 20, 100)
dummy_targets[0, 2, 5, 50] = 1.0  # One positive onset
dummy_targets[1, 0, 10, 30] = 1.0  # One positive onset

loss = loss_fn(dummy_logits, dummy_targets)
print(f'Test loss (mostly zeros target): {loss.item():.4f}')

# Verify: with all-zero predictions (logits << 0), loss should be HIGH
zero_logits = torch.full_like(dummy_logits, -10.0)  # sigmoid(-10) ≈ 0
loss_zero = loss_fn(zero_logits, dummy_targets)
print(f'Loss when model predicts all zeros: {loss_zero.item():.4f} (should be high!)')

In [ ]:
class GuitarSetDataset(Dataset):
    """Dataset for GuitarSet guitar transcription (v2 with augmentation)."""

    def __init__(self, data_dir, split='train', chunk_length_sec=3.0,
                 sample_rate=22050, augment=False):
        self.data_dir = Path(data_dir)
        self.chunk_length = chunk_length_sec
        self.sr = sample_rate
        self.hop_length = CONFIG['hop_length']
        self.augment = augment

        # Find audio and annotation files
        self.wav_files = sorted(self.data_dir.rglob('*.wav'))
        self.jams_files = sorted(self.data_dir.rglob('*.jams'))

        print(f"Found {len(self.wav_files)} WAV files")
        print(f"Found {len(self.jams_files)} JAMS files")

        # Build JAMS lookup by stem name
        self.jams_lookup = {}
        for jp in self.jams_files:
            self.jams_lookup[jp.stem] = jp
            self.jams_lookup[jp.stem.lower()] = jp

        self.samples = self._load_samples(split)
        print(f"Loaded {len(self.samples)} samples for {split}")
        if augment:
            print(f"  Data augmentation: ON (gain jitter, noise)")

    def _extract_track_id(self, wav_name):
        """Extract core track ID from WAV filename."""
        stem = wav_name
        for suffix in ['_mic', '_mix', '_hex_cln', '_hex']:
            if stem.endswith(suffix):
                stem = stem[:-len(suffix)]
                break
        for prefix in ['audio_mono-mic_', 'audio_mono-mic-',
                        'audio_mono-pickup_mix_', 'audio_mono-pickup_mix-',
                        'audio_hex-pickup_original_', 'audio_hex-pickup_original-',
                        'audio_hex-pickup_debleeded_', 'audio_hex-pickup_debleeded-']:
            if stem.startswith(prefix):
                stem = stem[len(prefix):]
                break
        return stem

    def _load_samples(self, split):
        samples = []
        unmatched = []

        for audio_path in self.wav_files:
            track_id = self._extract_track_id(audio_path.stem)
            jams_path = self.jams_lookup.get(track_id)
            if jams_path is None:
                jams_path = self.jams_lookup.get(track_id.lower())

            if jams_path is not None:
                samples.append({'audio': str(audio_path), 'jams': str(jams_path)})
            else:
                unmatched.append((audio_path.name, track_id))

        if len(samples) == 0:
            print(f"\nWARNING: No matched samples!")
            print(f"\nFirst 5 WAV -> track_id:")
            for name, tid in unmatched[:5]:
                print(f"  '{name}' -> '{tid}'")
            print(f"\nFirst 5 JAMS keys:")
            for k in list(self.jams_lookup.keys())[:5]:
                print(f"  '{k}'")
        elif unmatched:
            print(f"  ({len(unmatched)} WAV files had no matching JAMS)")

        np.random.seed(42)
        np.random.shuffle(samples)
        split_idx = int(len(samples) * 0.8)

        if split == 'train':
            return samples[:split_idx]
        else:
            return samples[split_idx:]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Load audio
        audio, sr = torchaudio.load(sample['audio'])
        if sr != self.sr:
            audio = torchaudio.functional.resample(audio, sr, self.sr)
        if audio.shape[0] > 1:
            audio = audio.mean(dim=0, keepdim=True)
        audio = audio.squeeze(0)

        # Random chunk
        chunk_samples = int(self.chunk_length * self.sr)
        if len(audio) > chunk_samples:
            start = np.random.randint(0, len(audio) - chunk_samples)
            audio = audio[start:start + chunk_samples]
            start_time = start / self.sr
        else:
            audio = F.pad(audio, (0, chunk_samples - len(audio)))
            start_time = 0

        # Convert to numpy for augmentation and CQT
        audio_np = audio.numpy()

        # v2: Data augmentation (training only)
        if self.augment:
            # Random gain (0.7x to 1.3x)
            if random.random() < 0.5:
                gain = random.uniform(0.7, 1.3)
                audio_np = audio_np * gain

            # Random noise (very small)
            if random.random() < 0.3:
                noise_level = random.uniform(0.001, 0.005)
                noise = np.random.randn(len(audio_np)).astype(np.float32) * noise_level
                audio_np = audio_np + noise

        # Compute CQT
        cqt = librosa.cqt(
            audio_np, sr=self.sr,
            hop_length=self.hop_length,
            n_bins=84, bins_per_octave=12
        )
        cqt = np.abs(cqt)
        cqt = torch.from_numpy(np.log(cqt + 1e-8)).float()

        # Load JAMS annotation
        jam = jams.load(sample['jams'])

        # Create targets
        num_frames = cqt.shape[-1]
        onset_target = torch.zeros(CONFIG['num_strings'], CONFIG['num_frets'], num_frames)
        frame_target = torch.zeros(CONFIG['num_strings'], CONFIG['num_frets'], num_frames)

        end_time = start_time + self.chunk_length

        for annot in jam.annotations:
            if annot.namespace == 'note_midi':
                for obs in annot.data:
                    if start_time <= obs.time < end_time:
                        midi_note = int(obs.value)
                        positions = midi_to_tab(midi_note)
                        if positions:
                            string_idx, fret = positions[0]
                            onset_frame = int((obs.time - start_time) * self.sr / self.hop_length)
                            end_frame = int((obs.time + obs.duration - start_time) * self.sr / self.hop_length)
                            if 0 <= onset_frame < num_frames:
                                onset_target[string_idx, fret, onset_frame] = 1
                            for f in range(max(0, onset_frame), min(num_frames, end_frame)):
                                frame_target[string_idx, fret, f] = 1

        return {'cqt': cqt, 'onsets': onset_target, 'frames': frame_target}


print('✅ GuitarSetDataset defined (v2 with augmentation)')

In [ ]:
class GuitarTabModel(nn.Module):
    """
    CRNN model for guitar tablature transcription.

    v2 change: forward() returns RAW LOGITS (no sigmoid).
    This is required for BCEWithLogitsLoss which is numerically stable
    and supports pos_weight.

    The state_dict is IDENTICAL to v1 (sigmoid has no learnable params),
    so trained checkpoints work directly with existing inference code.
    """

    def __init__(self, n_bins=84, num_strings=6, num_frets=20):
        super().__init__()

        self.num_strings = num_strings
        self.num_frets = num_frets

        # CNN encoder
        self.conv_stack = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),
            nn.Dropout(0.25),

            nn.Conv2d(32, 64, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),
            nn.Dropout(0.25),

            nn.Conv2d(64, 128, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),
            nn.Dropout(0.25),
        )

        self.flat_size = 128 * (n_bins // 8)

        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            self.flat_size, 256, num_layers=2,
            batch_first=True, bidirectional=True, dropout=0.3
        )

        # Output heads
        self.onset_head = nn.Linear(512, num_strings * num_frets)
        self.frame_head = nn.Linear(512, num_strings * num_frets)

    def forward(self, x):
        # x: (batch, n_bins, time)
        x = x.unsqueeze(1)  # Add channel dim

        # CNN
        x = self.conv_stack(x)  # (batch, 128, n_bins//8, time)

        # Reshape for LSTM: (batch, time, features)
        batch, channels, freq, time = x.shape
        x = x.permute(0, 3, 1, 2).reshape(batch, time, -1)

        # LSTM
        x, _ = self.lstm(x)

        # v2: Return RAW LOGITS (no sigmoid!)
        onset_logits = self.onset_head(x)
        frame_logits = self.frame_head(x)

        # Reshape to (batch, strings, frets, time)
        onset_logits = onset_logits.view(batch, time, self.num_strings, self.num_frets)
        onset_logits = onset_logits.permute(0, 2, 3, 1)

        frame_logits = frame_logits.view(batch, time, self.num_strings, self.num_frets)
        frame_logits = frame_logits.permute(0, 2, 3, 1)

        return onset_logits, frame_logits


# Test model
model = GuitarTabModel()
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

# Verify output is logits (can be negative, not bounded to 0-1)
dummy = torch.randn(2, 84, 100)
onset_out, frame_out = model(dummy)
print(f'Onset logits range: [{onset_out.min():.2f}, {onset_out.max():.2f}]')
print(f'Frame logits range: [{frame_out.min():.2f}, {frame_out.max():.2f}]')
print(f'Shape: {onset_out.shape}')  # (2, 6, 20, time)
print('✅ Model outputs raw logits (correct for BCEWithLogitsLoss)')

In [ ]:
# Create datasets with augmentation for training
train_dataset = GuitarSetDataset(DATA_DIR, split='train', augment=True)
val_dataset = GuitarSetDataset(DATA_DIR, split='val', augment=False)

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG['batch_size'],
    shuffle=True, num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=CONFIG['batch_size'],
    shuffle=False, num_workers=4
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

# Check class balance (this is what killed v1)
total_onset_pos = 0
total_onset_elements = 0
total_frame_pos = 0
total_frame_elements = 0

for i in range(min(20, len(train_dataset))):
    sample = train_dataset[i]
    total_onset_pos += sample['onsets'].sum().item()
    total_onset_elements += sample['onsets'].numel()
    total_frame_pos += sample['frames'].sum().item()
    total_frame_elements += sample['frames'].numel()

print(f"\n=== Class Balance (first 20 samples) ===")
onset_ratio = total_onset_pos / total_onset_elements * 100
frame_ratio = total_frame_pos / total_frame_elements * 100
print(f"Onset positive ratio: {onset_ratio:.4f}% ({total_onset_pos:.0f}/{total_onset_elements})")
print(f"Frame positive ratio: {frame_ratio:.4f}% ({total_frame_pos:.0f}/{total_frame_elements})")
print(f"\nThis is why v1 failed: plain BCELoss optimizes for the 99.97% zeros!")
print(f"v2 onset_pos_weight={CONFIG['onset_pos_weight']} compensates for this.")

In [ ]:
# Initialize model, optimizer, and v2 loss functions
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

model = GuitarTabModel(
    n_bins=84,
    num_strings=CONFIG['num_strings'],
    num_frets=CONFIG['num_frets']
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=10, factor=0.5, mode='min'
)

# v2: Weighted loss functions
onset_criterion = FocalBCEWithLogitsLoss(
    pos_weight=CONFIG['onset_pos_weight'],
    gamma=CONFIG['focal_gamma']
)
frame_criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([CONFIG['frame_pos_weight']]).to(device)
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Onset loss: FocalBCEWithLogitsLoss(pos_weight={CONFIG['onset_pos_weight']}, gamma={CONFIG['focal_gamma']})")
print(f"Frame loss: BCEWithLogitsLoss(pos_weight={CONFIG['frame_pos_weight']})")

In [ ]:
def train_epoch(model, loader, optimizer, onset_criterion, frame_criterion, device):
    model.train()
    total_loss = 0
    all_onset_logits = []
    all_onset_targets = []

    for batch in tqdm(loader, desc='Training'):
        cqt = batch['cqt'].to(device)
        onsets_target = batch['onsets'].to(device)
        frames_target = batch['frames'].to(device)

        optimizer.zero_grad()

        onset_logits, frame_logits = model(cqt)

        # Match time dimensions
        min_len = min(onset_logits.shape[-1], onsets_target.shape[-1])
        onset_logits = onset_logits[..., :min_len]
        frame_logits = frame_logits[..., :min_len]
        onsets_target = onsets_target[..., :min_len]
        frames_target = frames_target[..., :min_len]

        # v2: Weighted losses on logits
        onset_loss = onset_criterion(onset_logits, onsets_target)
        frame_loss = frame_criterion(frame_logits, frames_target)

        loss = onset_loss + frame_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        # Collect for F1 computation (subsample to save memory)
        all_onset_logits.append(onset_logits.detach().cpu())
        all_onset_targets.append(onsets_target.detach().cpu())

    # Compute epoch-level F1
    all_logits = torch.cat(all_onset_logits, dim=0)
    all_targets = torch.cat(all_onset_targets, dim=0)
    f1, prec, rec = compute_onset_f1(all_logits, all_targets)

    return total_loss / len(loader), f1, prec, rec


def validate(model, loader, onset_criterion, frame_criterion, device):
    model.eval()
    total_loss = 0
    all_onset_logits = []
    all_onset_targets = []

    with torch.no_grad():
        for batch in loader:
            cqt = batch['cqt'].to(device)
            onsets_target = batch['onsets'].to(device)
            frames_target = batch['frames'].to(device)

            onset_logits, frame_logits = model(cqt)

            min_len = min(onset_logits.shape[-1], onsets_target.shape[-1])

            onset_loss = onset_criterion(onset_logits[..., :min_len], onsets_target[..., :min_len])
            frame_loss = frame_criterion(frame_logits[..., :min_len], frames_target[..., :min_len])

            total_loss += (onset_loss + frame_loss).item()

            all_onset_logits.append(onset_logits[..., :min_len].cpu())
            all_onset_targets.append(onsets_target[..., :min_len].cpu())

    # Compute epoch-level F1
    all_logits = torch.cat(all_onset_logits, dim=0)
    all_targets = torch.cat(all_onset_targets, dim=0)
    f1, prec, rec = compute_onset_f1(all_logits, all_targets)

    return total_loss / len(loader), f1, prec, rec


print('✅ Training and validation functions defined')

In [ ]:
# ============================================================================
# TRAINING LOOP (v2 with F1 tracking and collapse detection)
# ============================================================================

SAVE_DIR = Path('/content/drive/MyDrive/tab_model_results_v2')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

best_val_loss = float('inf')
best_val_f1 = 0.0
history = {
    'train_loss': [], 'val_loss': [],
    'train_f1': [], 'val_f1': [],
    'train_precision': [], 'val_precision': [],
    'train_recall': [], 'val_recall': [],
}

# Collapse detection
zero_recall_count = 0
COLLAPSE_THRESHOLD = 10  # Stop if recall stays 0 for this many epochs

print("Starting training (v2 — weighted loss + F1 tracking)...")
print(f"Saving checkpoints to: {SAVE_DIR}")
print(f"Collapse detection: stop if onset recall = 0 for {COLLAPSE_THRESHOLD} epochs\n")

for epoch in range(CONFIG['num_epochs']):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{CONFIG['num_epochs']}")
    print(f"{'='*60}")

    train_loss, train_f1, train_prec, train_rec = train_epoch(
        model, train_loader, optimizer, onset_criterion, frame_criterion, device
    )
    val_loss, val_f1, val_prec, val_rec = validate(
        model, val_loader, onset_criterion, frame_criterion, device
    )

    scheduler.step(val_loss)

    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)
    history['train_precision'].append(train_prec)
    history['val_precision'].append(val_prec)
    history['train_recall'].append(train_rec)
    history['val_recall'].append(val_rec)

    lr = optimizer.param_groups[0]['lr']

    print(f"  Train — Loss: {train_loss:.4f} | Onset F1: {train_f1:.4f} | P: {train_prec:.4f} | R: {train_rec:.4f}")
    print(f"  Val   — Loss: {val_loss:.4f} | Onset F1: {val_f1:.4f} | P: {val_prec:.4f} | R: {val_rec:.4f}")
    print(f"  LR: {lr:.2e}")

    # Collapse detection
    if val_rec < 0.01:
        zero_recall_count += 1
        print(f"  ⚠️ Val onset recall near zero ({zero_recall_count}/{COLLAPSE_THRESHOLD})")
        if zero_recall_count >= COLLAPSE_THRESHOLD:
            print(f"\n🛑 TRAINING COLLAPSED: Onset recall has been ~0 for {COLLAPSE_THRESHOLD} epochs.")
            print(f"   Try increasing pos_weight or lowering learning rate.")
            break
    else:
        zero_recall_count = 0  # Reset counter

    # Save best model (by F1, not loss — loss can be misleading!)
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_f1': val_f1,
            'config': CONFIG,
        }, SAVE_DIR / 'best_tab_model.pt')
        print(f"  ✅ Saved best model (val_F1: {val_f1:.4f}, val_loss: {val_loss:.4f})")

    # Also save if best by loss (in case F1 metric is noisy early on)
    if val_loss < best_val_loss and val_f1 <= best_val_f1:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_f1': val_f1,
            'config': CONFIG,
        }, SAVE_DIR / 'best_tab_model_by_loss.pt')

    # Checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_f1': val_f1,
        }, SAVE_DIR / f'tab_model_epoch_{epoch+1}.pt')
        print(f"  💾 Checkpoint at epoch {epoch+1}")

print(f"\n{'='*60}")
print(f"Training complete!")
print(f"Best val F1: {best_val_f1:.4f}")
print(f"Best val loss: {best_val_loss:.4f}")
print(f"{'='*60}")

In [ ]:
# Plot training history (v2: includes F1 metrics)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(history['val_loss'], label='Val Loss', color='coral')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss (weighted BCE + Focal)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1
axes[1].plot(history['train_f1'], label='Train F1', color='steelblue')
axes[1].plot(history['val_f1'], label='Val F1', color='coral')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Onset F1')
axes[1].set_title('Onset F1 Score (THE metric that matters)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Precision/Recall
axes[2].plot(history['val_precision'], label='Val Precision', color='steelblue')
axes[2].plot(history['val_recall'], label='Val Recall', color='coral')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Score')
axes[2].set_title('Val Precision & Recall')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'training_history_v2.png', dpi=150)
plt.show()

print(f"\nFinal metrics:")
print(f"  Val F1: {history['val_f1'][-1]:.4f}")
print(f"  Val Precision: {history['val_precision'][-1]:.4f}")
print(f"  Val Recall: {history['val_recall'][-1]:.4f}")

## Training Complete!

Your trained guitar tablature model is saved to `Google Drive/tab_model_results_v2/`

**Key checkpoints:**
- `best_tab_model.pt` — Best model by onset F1 score
- `best_tab_model_by_loss.pt` — Best model by validation loss

**Deploy:** Copy `best_tab_model.pt` to `backend/models/pretrained/best_tab_model.pt`

The existing inference code (`guitar_tab_transcriber.py`) applies sigmoid at inference time,
so this checkpoint is a drop-in replacement — no code changes needed.